In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
df=pd.read_csv('/content/ganjoo_ipl_mbm_bowl_final.csv')


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14269 entries, 0 to 14268
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   date               14269 non-null  object 
 1   match_id           14269 non-null  int64  
 2   bowl_team          14269 non-null  object 
 3   bat_team           14269 non-null  object 
 4   bowler_id          14269 non-null  int64  
 5   bowler             14269 non-null  object 
 6   inns               14269 non-null  int64  
 7   overs              14269 non-null  object 
 8   runs               14269 non-null  int64  
 9   wickets            14269 non-null  int64  
 10  maiden             14269 non-null  int64  
 11  eco                14269 non-null  float64
 12  balls              14269 non-null  int64  
 13  lbw_or_bowled      14269 non-null  int64  
 14  dots               14269 non-null  int64  
 15  venue              14269 non-null  object 
 16  dots_per_ball_x    100

In [8]:
len(df['bowler_id'].unique()), len(df['match_id'].unique())


(471, 637)

In [10]:
df.sort_values(by='date', inplace=True)


In [17]:
# Group by 'bowler_id' to calculate dots/ball for each bowler
bowler_dots_per_ball = df.groupby('bowler_id')[['dots', 'balls']].sum().reset_index()
bowler_dots_per_ball['dots_per_ball'] = bowler_dots_per_ball['dots'] / bowler_dots_per_ball['balls']

# Drop the existing 'dots_per_ball' column if it already exists
if 'dots_per_ball' in df.columns:
    df.drop(columns=['dots_per_ball'], inplace=True)

# Merge the calculated dots/ball back into the original DataFrame
df = df.merge(bowler_dots_per_ball[['bowler_id', 'dots_per_ball']], on='bowler_id', how='left')

# Now, calculate the median of dots/ball for each bowler
median_dots_per_ball = df.groupby('bowler_id')['dots_per_ball'].transform('median')

# Fill null values in 'dots' column with (median of dots/balls for that bowler) * balls
df['dots'] = df['dots'].fillna(median_dots_per_ball * df['balls'])
df.head()

,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,overs,runs,wickets,maiden,eco,balls,lbw_or_bowled,dots,venue,dots_per_ball_x,dots_per_ball_y,dots_per_ball
0,01-04-2019,1175368,Delhi Capitals,Punjab Kings,45705,Colin Ingram,1,DNB,0,0,0,0.0,0,0,0,Punjab Cricket Association IS Bindra Stadium,NaN,NaN,NaN
1,01-04-2019,1175368,Delhi Capitals,Punjab Kings,931581,Rishabh Pant,1,DNB,0,0,0,0.0,0,0,0,Punjab Cricket Association IS Bindra Stadium,NaN,NaN,NaN
2,01-04-2019,1175368,Delhi Capitals,Punjab Kings,550215,Kagiso Rabada,1,4.0,32,2,0,8.0,24,0,11,Punjab Cricket Association IS Bindra Stadium,0.397260,0.397260,0.397260
3,01-04-2019,1175368,Punjab Kings,Delhi Capitals,422108,KL Rahul,2,DNB,0,0,0,0.0,0,0,0,Punjab Cricket Association IS Bindra Stadium,NaN,NaN,NaN
4,01-04-2019,1175368,Delhi Capitals,Punjab Kings,642519,Shreyas Iyer,1,DNB,0,0,0,0.0,0,0,0,Punjab Cricket Association IS Bindra Stadium,0.666667,0.666667,0.666667


In [27]:


# Replace 'DNB' with NaN and convert 'overs' to float
df['overs'] = df['overs'].replace('DNB', np.nan).astype(float)

# Fill NaN values in 'overs' with 0 (if needed)
df['overs'].fillna(0, inplace=True)

# Calculate dot ball points (+1 per 3 dot balls)
df['dot_ball_points'] = (df['dots'] // 3) * 1

# Calculate wicket points (+25 per wicket)
df['wicket_points'] = df['wickets'] * 25

# Calculate bonus for LBW/Bowled (+8 per dismissal)
df['bonus_lbw_bowled'] = df['lbw_or_bowled'] * 8

# Wicket milestone bonuses
df['4_wicket_bonus'] = ((df['wickets'] >= 4) & (df['wickets'] < 5)).astype(int) * 4
df['5_wicket_bonus'] = ((df['wickets'] >= 5) & (df['wickets'] < 6)).astype(int) * 8
df['6_wicket_bonus'] = (df['wickets'] >= 6).astype(int) * 12

# Maiden over bonus (+4 per maiden over)
df['maiden_over_bonus'] = df['maiden'] * 4

# Economy rate points calculation function
def calculate_economy_points(row):
    if row['overs'] < 5:  # Minimum 5 overs condition
        return 0
    eco = row['eco']
    if eco < 2.5:
        return 6
    elif 2.5 <= eco < 3.5:
        return 4
    elif 3.5 <= eco < 4.5:
        return 2
    elif 7 <= eco < 8:
        return -2
    elif 8.01 <= eco < 9:
        return -4
    elif eco >= 9:
        return -6
    return 0  # Default case

# Apply economy points calculation
df['economy_rate_points'] = df.apply(calculate_economy_points, axis=1)

# Total fantasy points calculation
df['bowling_fp'] = (
    df['dot_ball_points'] +
    df['wicket_points'] +
    df['bonus_lbw_bowled'] +
    df['4_wicket_bonus'] +
    df['5_wicket_bonus'] +
    df['6_wicket_bonus'] +
    df['maiden_over_bonus'] +
    df['economy_rate_points']
)

# Drop unnecessary columns
df.drop(columns=[
    'dot_ball_points', 'wicket_points', 'bonus_lbw_bowled',
    '4_wicket_bonus', '5_wicket_bonus', '6_wicket_bonus', 'maiden_over_bonus', 'economy_rate_points'
], inplace=True)

# Save the cleaned data
df.to_csv("processed_bowling_data.csv", index=False)

print("Processing complete")
df.tail()

<ipython-input-27-dc937bbd1917>:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['overs'].fillna(0, inplace=True)


Processing complete


,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,overs,runs,wickets,maiden,eco,balls,lbw_or_bowled,dots,venue,dots_per_ball_x,dots_per_ball_y,dots_per_ball,bowling_fp
14264,31-10-2020,1216535,Delhi Capitals,Mumbai Indians,28235,Shikhar Dhawan,2,0.0,0,0,0,0.000000,0,0,0,Dubai International Cricket Stadium,NaN,NaN,NaN,0
14265,31-10-2020,1216535,Delhi Capitals,Mumbai Indians,670025,Shimron Hetmyer,2,0.0,0,0,0,0.000000,0,0,0,Dubai International Cricket Stadium,NaN,NaN,NaN,0
14266,31-10-2020,1216535,Delhi Capitals,Mumbai Indians,642519,Shreyas Iyer,2,0.0,0,0,0,0.000000,0,0,0,Dubai International Cricket Stadium,0.666667,0.666667,0.666667,0
14267,31-10-2020,1216535,Mumbai Indians,Delhi Capitals,471342,Krunal Pandya,1,3.0,13,0,0,4.333333,19,0,9,Dubai International Cricket Stadium,0.322950,0.322950,0.322950,3
14268,31-10-2020,1216502,Sunrisers Hyderabad,Royal Challengers Bangalore,279810,Wriddhiman Saha,1,0.0,0,0,0,0.000000,0,0,0,Sharjah Cricket Stadium,NaN,NaN,NaN,0


In [31]:
df = df[['bowler_id', 'match_id', 'bowling_fp']]
df.to_csv('Bowling_fantasyPoints.csv', index=False)